In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# Z3-Solver

In TypeScript, there is no direct equivalent to pycosat, but a great alternative is the theorem prover [Z3](https://github.com/Z3Prover/z3) with the npm package `z3-solver`.

In [ ]:
import { init } from "z3-solver";

In the TypeScript version, the [Z3 Solver](https://github.com/Z3Prover/z3) is used, which is a powerful SMT solver capable of handling more complex logical formulas than basic SAT solvers like PicoSat.

- **Boolean variables** are declared using `ctx.Bool.const` with the variable name as a string, e.g., `p`, `q`, `r`.
- **Clauses** are constructed using logical operations like `ctx.Or` for disjunction and `ctx.Not` for negation.
- A formula in conjunctive normal form (CNF) is composed with `ctx.And` combining multiple clauses.
- The solver is initialized with `new ctx.Solver()`, and the formula is added via `solver.add(formula)`.
- The satisfiability check is performed asynchronously with `await solver.check()`.
- If the formula is satisfiable (`"sat"`), the model can be retrieved with `solver.model()` and printed.

Finally, the formula 
$$ (p \vee \neg q \vee r) \wedge (\neg p \vee q \vee \neg r) \wedge (\neg p \vee \neg q \vee r) \wedge (p \vee q \vee \neg r), $$
which is in conjunctive normal form, is represented in the following code:

In [ ]:
import { init } from "z3-solver";

async function runExample() {
  const Z3 = await init();
  const ctx = Z3.Context("");
  const p = ctx.Bool.const("p");
  const q = ctx.Bool.const("q");
  const r = ctx.Bool.const("r");
  const clause1 = ctx.Or(p, ctx.Not(q), r);
  const clause2 = ctx.Or(ctx.Not(p), q, ctx.Not(r));
  const clause3 = ctx.Or(ctx.Not(p), ctx.Not(q), r);
  const clause4 = ctx.Or(p, q, ctx.Not(r));
  const f = ctx.And(clause1, clause2, clause3, clause4);
  const solver = new ctx.Solver();
  solver.add(f);
  const result = await solver.check();
  console.log(result.toString());
    if (result.toString() === "sat") {
    const model = solver.model();
    console.log("Model:");
    console.log(model.toString());
  }
}
await runExample();

This shows that the formula `f` is satisfiable and that the  propositional valuation 
$$ \mathcal{I} = \{ p \mapsto \texttt{False}, q \mapsto \texttt{False}, r \mapsto \texttt{False} \} $$
is a solution for `f`, i.e. we have
$$ \mathcal{I}(\texttt{f}) = \texttt{True}. $$

## Transforming Clauses into Z3-Solver Format

In [ ]:
type Variable = string;
type Literal = Variable | ['¬', Variable];
type Clause = Set<Literal>;

In order to use *Z3-Solver*, we need a function that transforms a formula that is in conjunctive normal form
into the format of *Z3-Solver*.  Furthermore, we need a function that can translate a solution found by *Z3-Solver* back into our format.

The function `findVariables` takes a set of `Clauses` and returns the set of all propositional variables
occurring in this set.  

In [ ]:
function findVariables(Clauses: Clause[]): Set<Variable> {
  const Variables = new Set<Variable>();
  for (const Clause of Clauses) {
    for (const literal of Clause) {
      if (Array.isArray(literal) && literal[0] === '¬') {
        Variables.add(literal[1]);
      } else if (typeof literal === 'string') {
        Variables.add(literal);
      }
    }
  }
  return Variables;
}

The function `numberVariables(Clauses)` takes a set of `Clauses` as input.  It returns two dictionaries:
* The Map `Var2Int` maps every propositional variable occurring in `Clauses` to a unique natural number.
* The Map `Int2Var` is the mapping that is inverse to the dictionary `Var2Int`.

In [ ]:
function numberVariables(Clauses: Clause[]): [Map<Variable, number>, Map<number, Variable>] {
  const Variables = findVariables(Clauses);
  let count = 1;
  const Var2Int = new Map<Variable, number>();
  const Int2Var = new Map<number, Variable>();
  for (const variable of Variables) {
    Var2Int.set(variable, count);
    Int2Var.set(count, variable);
    count++;
  }
  return [Var2Int, Int2Var];
}

The function `literal2int` takes a literal and transforms this literal into an integer
representing the literal.  If the literal is a negated variable, the integer is negative, else it is positive.
`Var2Int` is a dictionary mapping propositional variables to natural numbers.

In [ ]:
function literal2int(literal: Literal, Var2Int: Map<Variable, number>): number {
  if (Array.isArray(literal) && literal[0] === '¬') {
    return -Var2Int.get(literal[1])!;
  } else if (typeof literal === 'string') {
    return Var2Int.get(literal)!;
  } else {
    throw new Error("Invalid literal format");
  }
}

The function `clause2pyco(Clause, Var2Int)` transforms a set of literals into a list of integers.
`Var2Int` is a dictionary mapping the propositional variables to integers.

In [ ]:
function clause2pyco(Clause: Set<Literal>, Var2Int: Map<Variable, number>): number[] {
  return Array.from(Clause).map(literal => literal2int(literal, Var2Int));
}

The function `clauses2pyco(Clauses, Var2Int)` transforms a set of `Clauses` into a list of lists of integers.
`Var2Int` is a dictionary mapping the propositional variables to integers.

In [ ]:
function clauses2pyco(Clauses: Set<Set<Literal>>, Var2Int: Map<Variable, number>): number[][] {
  return Array.from(Clauses).map(clause => clause2pyco(clause, Var2Int));
}

The function `int2var(Numbers, Int2Var)` takes a list of numbers representing a set of literals
and returns the associated list of literals.

In [ ]:
function int2var(Numbers: number[], Int2Var: Map<number, Variable>): Set<Set<Literal>> {
  const Result = new Set<Set<Literal>>();
  for (const n of Numbers) {
    if (n > 0) {
      Result.add(new Set<Literal>([Int2Var.get(n)!]));
    } else {
      Result.add(new Set<Literal>([['¬', Int2Var.get(-n)!]]));
    }
  }
  return Result;
}

## Sudoku

The finish mathematician Arto Inkala claims to have created the [hardest sudoku](https://abcnews.go.com/blogs/headlines/2012/06/can-you-solve-the-hardest-ever-sudoku) ever.  It is defined below.

In [ ]:
function createPuzzle(): (number | string)[][] {
  return [
    [ 8 , "*", "*", "*", "*", "*", "*", "*", "*"], 
    ["*", "*",  3 ,  6 , "*", "*", "*", "*", "*"],
    ["*",  7 , "*", "*",  9 , "*",  2 , "*", "*"],
    ["*",  5 , "*", "*", "*",  7 , "*", "*", "*"],
    ["*", "*", "*", "*",  4 ,  5 ,  7 , "*", "*"],
    ["*", "*", "*",  1 , "*", "*", "*",  3 , "*"],
    ["*", "*",  1 , "*", "*", "*", "*",  6 ,  8 ],
    ["*", "*",  8 ,  5 , "*", "*", "*",  1 , "*"],
    ["*",  9 , "*", "*", "*", "*",  4 , "*", "*"]
  ];
}

We will solve this Sudoku using the Davis-Putnam algorithm.  We use the following variables:
* `Q<r,c,d>` is a Boolean variable stating that the field in row `r` and column `c` holds the digit `d`.
  Here, `r`, `c`, `d` are all elements from the set $\{1,\cdots,9\}$.
    
The function `var(row, col, digit)` returns a formated string that is interpreted as a variable name.

In [ ]:
function varName(row: number, col: number, digit: number): string {
    return `Q<${row},${col},${digit}>`;
}

In [ ]:
varName(1,2,3);

The function `atMostOne(S)` takes a set `S` of propositional variables as its argument.  It returns a set of clauses
expressing the fact that at most one of the variables of `S` is true.

In [ ]:
function atMostOne(S: Set<Variable>): Set<Clause> {
    const result = new Set<Clause>();
    const arr = Array.from(S);
    for (let i = 0; i < arr.length; i++) {
        for (let j = i + 1; j < arr.length; j++) {
            const p = arr[i];
            const q = arr[j];
            result.add(new Set<Literal>([['¬', p], ['¬', q]]));
        }
    }
    return result;
}

The function `atLeastOne(S)` takes a set `S` of propositional variables as its argument.  It returns a set of clauses
expressing the fact that at least one of the variables of `S` is true.

In [ ]:
function atLeastOne(S: Set<Variable>): Set<Clause> {
    return new Set([new Set(S)]);
}

The function `exactlyOne(S)` takes a set `S` of propositional variables as its argument.  It returns a set of clauses
expressing the fact that exactly one of the variables of `S` is true.

In [ ]:
function exactlyOne(S: Set<Variable>): Set<Clause> {
    const atMost = atMostOne(S);
    const atLeast = atLeastOne(S);
    return new Set([...atMost, ...atLeast]);
}

In [ ]:
exactlyOne(new Set(['a', 'b', 'c']));

The function `exactlyOnce` takes a list `L` of pairs of indices as its argument.  The elements of `L` are pairs of the form
`(row, col)`, where both `row` and `col` are elements of the set $\{1, \cdots, 9\}$.
It returns a set of formulas expressing that all Sudoku fields specified by the coordinate pairs in `L` take different digits as values.

In [ ]:
function exactlyOnce(L: Array<[number, number]>): Set<Clause> {
    const Clauses = new Set<Clause>();
    for (let digit = 1; digit <= 9; digit++) {
        const vars = new Set<Variable>(
            L.map(([row, col]) => varName(row, col, digit))
        );
        const exact = exactlyOne(vars);
        for (const clause of exact) {
            Clauses.add(clause);
        }
    }
    return Clauses;
}

In [ ]:
const result = exactlyOnce(
    Array.from({ length: 9 }, (_, i) => [1, i + 1] as [number, number])
);
console.log(result.size);
for (const clause of result) {
    console.log(`{${[...clause].map(lit => JSON.stringify(lit)).join(', ')}}`);
}

The function `exactlyOneDigit(row, col)` takes integers `row` and `col` as arguments.  These specify the row and column of a field in a Sudoku.  The function returns a set of clauses specifying that exactly one of the variables

* `Q<row,col,1>`, `Q<row,col,2>`, $\cdots$, `Q<row,col,9>`

is `True`.

In [ ]:
function exactlyOneDigit(row: number, col: number): Set<Clause> {
    const vars = new Set<Variable>();
    for (let digit = 1; digit <= 9; digit++) {
        vars.add(varName(row, col, digit));
    }
    return exactlyOne(vars);
}

In [ ]:
exactlyOneDigit(1, 1);

The function `constraintsFromPuzzle`  returns a set of clauses stating that the variables corresponding to numbers that are already given in the Sudoku puzzle take the values that are specified.

In [ ]:
function constraintsFromPuzzle(): Set<Clause> {
    const Puzzle = createPuzzle();
    const Variables: Variable[] = [];
    for (let row = 0; row < 9; row++) {
        for (let col = 0; col < 9; col++) {
            if (Puzzle[row][col] !== '*') {
                Variables.push(varName(row + 1, col + 1, Puzzle[row][col] as number));
            }
        }
    }
    return new Set(Variables.map(v => new Set<Literal>([v])));
}

In [ ]:
constraintsFromPuzzle();

The function `allConstraints` returns a CSP that encodes the given sudoku as a CSP.

In [ ]:
function literalKey(lit: Literal): string {
    if (typeof lit === 'string') return lit;
    return `¬${lit[1]}`;
}

function equalClauses(c1: Clause, c2: Clause): boolean {
    if (c1.size !== c2.size) return false;
    for (const lit of c1) {
        let found = false;
        for (const l2 of c2) {
            if (literalKey(l2) === literalKey(lit)) {
                found = true;
                break;
            }
        }
        if (!found) return false;
    }
    return true;
}

function addClauseUnique(Clauses: Set<Clause>, clause: Clause) {
    for (const c of Clauses) {
        if (equalClauses(c, clause)) return; // Clause already present
    }
    Clauses.add(clause);
}

function allConstraints(): Set<Clause> {
    const L = [1, 2, 3, 4, 5, 6, 7, 8, 9];
    let Clauses = constraintsFromPuzzle();

    // Exactly one digit in every cell
    for (const row of L) {
        for (const col of L) {
            for (const clause of exactlyOneDigit(row, col)) {
                addClauseUnique(Clauses, clause);
            }
        }
    }

    // Unique entries in each row
    for (const row of L) {
        for (const clause of exactlyOnce(L.map(col => [row, col] as [number, number]))) {
            addClauseUnique(Clauses, clause);
        }
    }

    // Unique entries in each column
    for (const col of L) {
        for (const clause of exactlyOnce(L.map(row => [row, col] as [number, number]))) {
            addClauseUnique(Clauses, clause);
        }
    }

    // Unique entries in each 3x3 square
    for (let r = 0; r < 3; r++) {
        for (let c = 0; c < 3; c++) {
            const blockCells: Array<[number, number]> = [];
            for (let row = 1; row <= 3; row++) {
                for (let col = 1; col <= 3; col++) {
                    blockCells.push([r * 3 + row, c * 3 + col]);
                }
            }
            for (const clause of exactlyOnce(blockCells)) {
                addClauseUnique(Clauses, clause);
            }
        }
    }

    return Clauses;
}

In [ ]:
const clauses = Array.from(allConstraints());

for (const clause of clauses) {
    if (clause.size === 1) {
        console.log(clause);
    }
}

for (const clause of clauses) {
    if (clause.size === 9) {
        console.log(clause);
    }
}

In [ ]:
clauses.length

This function abstracts solving Sudoku by encoding logical constraints and querying the SMT solver for a valid assignment of Boolean variables that satisfy all Sudoku rules.

In [ ]:
import { Model } from "z3-solver";

async function sudoku(): Promise<Model<""> | null> {
  const Z3 = await init();
  const ctx = Z3.Context("");

  const ClausesSet: Set<Set<Literal>> = allConstraints();

  const clausesZ3 = Array.from(ClausesSet).map(clause => {
    const literalsZ3 = Array.from(clause).map(lit => {
      if (Array.isArray(lit) && lit[0] === '¬') {
        const v = ctx.Bool.const(lit[1]);
        return ctx.Not(v);
      } else if (typeof lit === 'string') {
        return ctx.Bool.const(lit);
      } else {
        throw new Error('Invalid literal format');
      }
    });
    return ctx.Or(...literalsZ3);
  });

  const formula = ctx.And(...clausesZ3);

  const solver = new ctx.Solver();
  solver.add(formula);

  const result = await solver.check();

  if (result.toString() === "sat") {
    const model = solver.model();
    console.log("Solution found.");
    return model;
  } else {
    console.log("The problem is not solvable!");
    return null;
  }
}

In [ ]:
9**3

Even though this Sudoku is modelled using $9^3 = 729$ propositional variables and we have 10551 clauses, Z3-Solver uses about 15 seconds to solve the problem. 

In [ ]:
console.time("sudoku");
const Solution = await sudoku();
console.timeEnd("sudoku");

In [ ]:
Solution.toString();

The function `removeNegative` removes the negative literals from the given solution `S` and returns the set of variables that have to be `True`.  

In [ ]:
function removeNegative(modelStr: string): Set<string> {
  const literals = new Set<string>();
  const regex = /\(define-fun \|([^|]+)\| \(\) Bool\s+(true|false)\)/g;
  let match;
  while ((match = regex.exec(modelStr)) !== null) {
    const varName = match[1];  
    const value = match[2];  
    if (value === "true") {
      literals.add(varName);
    }
  }
  return literals;
}

In [ ]:
const SolutionNoNegative = removeNegative(Solution.toString());

In [ ]:
SolutionNoNegative

## Graphical Representation

In [ ]:
function showSolution(Solution: Set<string>, width: string = '50%'): void {
  const Sudoku = createPuzzle();
  const solutionMap: Record<string, number> = {};
  for (const literal of Solution) {
    const match = literal.match(/^Q<(\d),(\d),(\d)>$/);
    if (match) {
      const row = parseInt(match[1], 10) - 1;
      const col = parseInt(match[2], 10) - 1;
      const digit = parseInt(match[3], 10);
      solutionMap[`V${row + 1}${col + 1}`] = digit;
    }
  }
  for (let row = 0; row < 9; row++) {
    for (let col = 0; col < 9; col++) {
      if (Sudoku[row][col] !== '*') {
        delete solutionMap[`V${row + 1}${col + 1}`];
      }
    }
  }
  let html = `<table style="width:${width}; border-collapse: collapse; border: 2px solid black;">`;
  for (let row = 0; row < 9; row++) {
    html += '<tr>';
    for (let col = 0; col < 9; col++) {
      const key = `V${row + 1}${col + 1}`;
      const value = solutionMap[key] || Sudoku[row][col];
      const boxColor = Math.floor(row / 3) % 2 === Math.floor(col / 3) % 2 ? '#f0f0f0' : '#ffffff';
      html += `<td style="border:1px solid #888; width:30px; height:30px; text-align:center; font-size:16px; background-color:${boxColor};">${value}</td>`;
    }
    html += '</tr>';
  }
  html += '</table>';
  tslab.display.html(html);
}

In [ ]:
showSolution(SolutionNoNegative);